<a href="https://colab.research.google.com/github/CDPHE-bioinformatics/.github/blob/main/COVID_QC_Seq_Results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COVID QC NOTEBOOK
![alt text](https://i0.wp.com/roomescapeartist.com/wp-content/uploads/2019/01/quality-cat-memes.gif?fit=300%2C188&ssl=1)

<br>

## ⏳ Overview

>This notebook is intended to be used for post analysis data processing and QC of covid clinical sequencing data.

<br>

## 🔗 Links

> *   [Google sheet: `covid_seq_qc`](https://docs.google.com/spreadsheets/d/1MXLpW5UL6oSeNM0AWFAXfI204nwULjqExzW9GlWgAY0/edit?gid=0#gid=0) (ask Molly for sharing permissions if you are unable to access)
> * [Tableau Datasource: `covid_seq_qc` ](https://cohealthviz.dphe.state.co.us/#/site/LabSecure/datasources/140702/askData)
> * [Tableau View: Percent Coverage Boxplot](https://cohealthviz.dphe.state.co.us/#/site/LabSecure/views/covid_sequencing_qc/percent_coverage_boxplot?:iid=3)
> * [Google Drive: cov sequencing results](https://drive.google.com/drive/folders/1DOyms33NMdslNGs2K1vzWqP6SV6d3Sny)
> * [Google sheet: `manual_sample_exclusions`](https://docs.google.com/spreadsheets/d/1qj0a9kzbyXkiaEnjJOtCkILi68lK9AdjytIvQ-m4E5w/edit?gid=0#gid=0)
> * `s_gene_qc_truth_table` [google sheet](https://docs.google.com/spreadsheets/d/10mlS5me_lUbvmAFEVKcgwRmUQptsX9mHrP1LXSXYwms/edit?gid=874459759#gid=874459759)

<br>

## 🔐 Initial setup - Colab secrets
> To manage your Tableau key:
 > 1.  Your Tableau Personal Access Token (PAT) will expire after 15 days of inactivity, so you may need to generate new ones occasionally and update them in the colab notebook.
  2. Go to Tableau's website and go into "My Account Settings" listed under your initials logo in the upper right hand corner.
  3. Under Personal Access Tokens, enter a descriptive token name (e.g. "colab") and click "Create Token". You will be shown a secret; this secret will only be shown to you once, so be sure to save/store it in colab immediately.
  4. In colab, click on the 🔑 icon in the left-side panel and store the following key-value pairs:
  ```
  tableau_token_name : <name chosen for token in Tableau, e.g. "colab">
  tableau_token_value : <PAT generated>
  ```

<br>

##  📃 Directions
>- Run each block of code using the ▶ to the left of the block.
>- User input: Select values from the drop down menus or enter comma separated lists where indicated.  
>- ✋✅ - Indicate manual checkpoints. These steps require review of output.
>- 👾 - Indicate code blocks to run.
>
>
> The notebook is divided into 3 sections as follows:
> ### 🕐 Section 1: General Set Up
 **This section should be completed by the analyst and rerun by the QC checker.** In this section, you will establish colab notebook secrets, load python modules, authenicate your google user credentials, set file paths, and declare a list of project names.
> ### 🕑 Section 2:  QC performed by Analyst
**This section should be completed by the analyst and rerun by the QC checker.** In this section, you will download the summary results files from the GCP bucket, check for PCR/sequencing inconsistencies, and summarize results for posting in the GSP chat.
> ### 🕒 Section 3: QC performed by QC checker
**This section should be completed by QC checker.** In this section, you will review the QC performed by the analyst, push QC reports to the GCP project bucket, generate the parsing files for Horizon, delete fastqs for failed projects, add failed samples to the `manual_sample_exclusions` sheet, and add the analyst name to submissions ready list.

<br>

## ❓ What makes a QC Pass?
> Use the table below to determine if the sample and/or plate passed QC. This table applies to most use cases, but there may be exceptions.
>
>| type | QC pass description |
>|-----------|---------------------|
>|positive control|(percent coverage > 80% OR >50% samples pass QC) AND <br> lineage matches reference sequence used during sequencing|
>|negative control|percent coverage < 10%|
>|verification|TBD- no verification samples are currently used during analysis|
>|sample|percent coverage > 90% AND<br> no PCR/sequencing mismatch (or mismatch can't be confirmed)|
>|plate|positive and negative controls pass QC AND<br> percent coverage boxplots show most samples have > 80% percent coverage AND<br> the majority of samples on plate have > 90% AND<br> there is no suspicion of PCR contamination or plate orientation flip|
>
> <br>
>
> What do if a sample and/or plate fails:
* If the positive control, negative control, or verificaton samples fail, then the whole plate is failed and should be re-sequenced.
* If the majority of the samples on the plate have less than 90% coverage or there is a large spread shown in the coverage boxplots, then the whole plate is failed and should be re-sequenced.
* If the majority of the samples on the plate have PCR/sequencing mismatches, then this could indicate a plate orientation issue or a PCR contamination issue. Each issue should be investigated before deciding on QC status.
* If a few samples on the plate have PCR/sequencing mismatches, then each sample should be investigated. If the mismatch is confirmed then the sample should be failed and re-sequenced.

<br>

---

# 🕐 Section 1: General Set Up

**This section should be completed by the analyst and rerun by the QC checker**

In [ ]:
# @title 👾 Import libraries and set workspace variables

### import python modules
import os
import re
import shutil
import numpy as np
import pickle # for grabbing lineges from old qc sheet; stored in a pickle
import time
from datetime import date, timedelta
import glob
import gspread
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from google.auth import default
from google.colab import auth, drive, userdata
from dataclasses import dataclass
from typing import Optional
import ipywidgets as widgets
from IPython.display import display, Javascript
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import json

### install tableau requirements
!pip install -q tableauserverclient
import tableauserverclient as TSC

### authenticate google sheets credentials
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

#### mount google drive
drive.mount('/content/drive')

# set tableau variables
sitename = 'LabSecure'
server = TSC.Server('https://cohealthviz.dphe.state.co.us/', use_server_version = True)
tableau_token_name = userdata.get('tableau_token_name')
tableau_token_value = userdata.get('tableau_token_value')
tableau_auth = TSC.PersonalAccessTokenAuth(tableau_token_name, tableau_token_value, sitename)
# see tips and tricks colab notebook for how to get these names and ids
# also see end of notebook
datasource_name = 'covid_seq_qc (covid_seq_qc)' # datasource name as appears on tableau
datasource_id = 'cc2046d5-790a-42bb-8262-88c9d1210405' # name = 'covid_seq_qc (covid_seq_qc)'
workbook_id = '0b7011dc-3b4c-4513-a9e8-e83b5d02e50e' # name = 'covid_sequencing_qc'
rt_pcr_view_id = '4a0c74bd-8daa-45de-ad27-803867648553' # name = 'rt-pcr_results_merged_with_seq_qc_table'
boxplot_view_id = 'b41b41a1-2b65-4c46-aeae-70796443310d' # name = percent_coverage_boxplot


# set paths
#### set local paths
results_dir = '/content/results/'
qc_dir = '/content/qc/'
bam_dir = '/content/bam/'
horizon_report_dir = '/content/horizon_report/'
tableau_dir = '/content/tableau/'
samtools_depth_dir = '/content/samtools_depth/'

dirs = [results_dir, qc_dir, bam_dir, horizon_report_dir, tableau_dir, samtools_depth_dir]
for dir in dirs:
  os.makedirs(dir, exist_ok = True)

#### set timezone to Mountain
!rm /etc/localtime
!ln -s /usr/share/zoneinfo/US/Mountain /etc/localtime
!date

#### set colors
# these colors are used for warnings in GSP chat printout
class Colors:
  OKGREEN = '\033[92m'
  WARNING = '\033[93m'
  RED = '\033[91m'
  ENDC = '\033[0m'
  BOLD = '\033[1m'

today = str(date.today())

#### silence warnings
pd.options.mode.chained_assignment = None

# Function to print larger
def increase_font(font_size: int) -> None:
  font_size = str(font_size) + 'px'
  display(Javascript('''
  for (rule of document.styleSheets[0].cssRules){
    if (rule.selectorText=='body') {
      rule.style.fontSize = "%s"
      break
    }
  }
  ''' % font_size))

#### sequencing results dtype
# maybe reduce this list to just the ones we use
# remember this list will be shorted when we streamline the results file
# with the next version of the workflow
sequencing_results_dtype = {
  'hsn': object,
  'sample_name': object,
  'project_name': object,
  'plate_name': object,
  'run_name': object,
  'analysis_date': object,
  'run_date': object,
  'assembly_pass': 'boolean',
  'percent_coverage': 'float64',
  'nextclade': object,
  'pangolin_lineage': object,
  'expanded_lineage': object,
  'aggregated_lineage': object,
  'spike_mutations': object,
  'N_bases': 'Int64',
  'aligned_bases': 'Int64',
  'download_date': object,
  'fasta_header': object,
  'fastq_dir': object,
  'guppy_version': object,
  'index_1': object,
  'index_1_id': object,
  'index_2': object,
  'index_2_id': object,
  'index_kit': object,
  'index_position': 'Int64',
  'index_set': 'Int64',
  'index_well': object,
  'instrument_id': object,
  'library_prep': object,
  'mapped_reads': 'Int64',
  'mean_base_quality': 'float64',
  'mean_depth': 'float64',
  'mean_map_quality': 'float64',
  'medaka_version': object,
  'nextclade_version': object,
  'non_ambiguous_bases': 'Int64',
  'note': object,
  'organism': object,
  'out_dir': object,
  'pangoLEARN_conflict': 'float64',
  'pango_designation_version': object,
  'pangolin_constellation_version': object,
  'pangolin_is_designated': 'boolean',
  'pangolin_note': object,
  'pangolin_qc_notes': object,
  'pangolin_qc_status': object,
  'pangolin_scorpio_call': object,
  'pangolin_scorpio_conflict': 'float64',
  'pangolin_scorpio_support': object,
  'pangolin_scorpio_version': object,
  'pangolin_version': object,
  'platform': object,
  'primer_set': object,
  'project_type': object,
  'read_length': 'Int64',
  'read_type': object,
  'sample_type': object,
  'sample_well': object,
  'tag': object,
  'terra_data_table_path': object,
  'total_AA_deletions': 'Int64',
  'total_AA_substitutions': 'Int64',
  'total_nucleotide_deletions': 'Int64',
  'total_nucleotide_insertions': 'Int64',
  'total_nucleotide_mutations': 'Int64',
  'workbook_path': object,
}

# set up project dataclass
@dataclass
class Project:
  """
  :param name: the project name
  :param results_df: the qc results as a DataFrame
  :property terra_outputs: the terra outputs bucket path
  :property results_dir: the results_dir in this colab notebook
  """
  name: str
  results_df: Optional[pd.DataFrame] = None

  @property
  def terra_outputs(self) -> str:
    return os.path.join(base_gcp_path_pull, self.name, 'terra_outputs')

  @property
  def raw_fastqs_gcp(self) -> str:
    return os.path.join(base_gcp_path_pull, self.name, 'fastq_pass')

  @property
  def scrubbed_fastqs_gcp(self) -> str:
    return os.path.join(self.terra_outputs, 'fastq_scrubbed')

  @property
  def sequencing_results_gcp(self) -> str:
    return os.path.join(self.terra_outputs,
                        f'summary_results/{self.name}_sequencing_results*.csv')

  @property
  def sequencing_results_local(self) -> str:
    glob_path = os.path.join(results_dir,
                            self.name,
                            f'{self.name}_sequencing_results*.csv')
    return glob.glob(glob_path)[0]

  @property
  def results_dir(self) -> str:
    return os.path.join(results_dir, self.name)


In [ ]:
# @title 👾 Select start up variables - 1/2 { run: "auto" }
# Jupyter widgets for dynamic options

# Commenting out the repo version block for now since we don't need it yet
"""
# Grab SC2 repo version options and display in widget
version_options = !git ls-remote --tags --sort=-v:refname https://github.com/CDPHE-bioinformatics/CDPHE-SARS-CoV-2 | awk '{print $2}' | awk -v FS="/" '{print $3}'
version_widget = widgets.Dropdown(
    options = version_options,
    value = version_options[0],
    description='CDPHE-SARS-CoV-2 repo version:',
    disabled=False,
    style = {'description_width' : 'initial'}
)
display(version_widget)
"""

# Display selection widget of available projects from run tracker - those that
# have been run but not uploaded and excluding qc fails
run_tracker = gc.open('run_tracker').worksheet('cov')
df = pd.DataFrame.from_records(run_tracker.get('C:J'))
df.columns = df.iloc[0]
df = df.iloc[1:]
df = df[df.qc_status.isna()]
project_query = 'ran == "TRUE" & uploaded == "FALSE"'
project_options = df.query(project_query).project_name.to_list()

project_widget = widgets.SelectMultiple(
  options=project_options,
  description='Project options:',
  disabled=False,
  style = {'description_width' : 'initial'}
)
display(project_widget)


In [ ]:
# @title 👾✋✅ Select start up variables - 2/2 { run: "auto" }

"""
version_widget.close()
version_chosen = version_widget.value
workflow_version = version_chosen.replace('.', '_')
"""
# colab form selections for static options
notebook_run_type = 'Production' #@param ['Production', 'Development and Testing']
role = 'QC checker' #@param ['Analyst', 'QC checker']
analyst = 'Laura' # @param ['Alena', 'Arianna', 'Daniel', 'Emily', 'Laura', 'Molly', 'Sam']
qc_checker = 'Molly' #@param ['Laura', 'Molly', 'Sam']
projects_to_add = '' # @param {type:"string", placeholder:"Any any projects not selected from above. Enter as a comma separated list"}

if projects_to_add != '':
  projects_to_add = [project.strip() for project in projects_to_add.split(',')]
else: projects_to_add = []

#### set google drive and GCP paths
shared_drive_path = '/content/drive/Shareddrives/CDPHE LSD Sequencing'
if notebook_run_type == 'Production':
  base_gdrive_path = f'{shared_drive_path}/COVID/Sequencing Run Results/'
  base_gcp_path_pull = base_gcp_path_push = 'gs://seq_projects/cov/'
  sample_exclusions = 'manual_sample_exclusions'
elif notebook_run_type == 'Development and Testing':
  base_gdrive_path = f'{shared_drive_path}/Bioinformatics/testing/cov_qc_notebook/'
  base_gcp_path_pull = 'gs://seq_projects/cov/'
  base_gcp_path_push = 'gs://bioinformatics-sandbox/notebook_testing/covid_qc_notebook/'
  sample_exclusions = 'manual_sample_exclusions_DEVELOPMENT'

results_drive_path = os.path.join(base_gdrive_path, 'cov')
horizon_drive_path = os.path.join(base_gdrive_path, 'wgs_horizon_reports')

# get all projects and create Project instances
project_widget.close()
projects_chosen = list(project_widget.value)
projects = projects_chosen + projects_to_add
increase_font(18)

# declare projects as Project class
project_list = []
for project in projects:
  p = Project(project)
  project_list.append(p)

# Print widget values chosen
"""print(f'Repo version: {version_chosen}\n')"""
print('Projects:', *(p for p in projects), sep = '\n')

---
# 🕑 Section 2 : QC check by Analyst

**This section should be completed by the analyst.** In this section, you will download the summary results files from the GCP bucket, check for PCR/Sequencing inconsistencies, and summarize results for posting in the GSP chat.




In [ ]:
#@title 👾✋✅ Append sequencing results data to covid_seq_qc google sheet, refresh Tableau extract, and download boxplots
# @markdown Run this cell and examine the boxplots, which show the distribution
# @markdown of the percent coverage of samples with less than 25 CT for each
# @markdown project. Boxplots should be tight and have a median >80%.
# @markdown
# @markdown **If the boxplot shows a large spread and/or the median is <80%,
# @markdown you should report the run to the QC checker as a
# @markdown possible run failure.**
# @markdown
# @markdown A web interface version of the boxplots can be found
# @markdown [here](https://cohealthviz.dphe.state.co.us/#/site/LabSecure/views/covid_sequencing_qc/percent_coverage_boxplot?:iid=3).


def download_summary_results(project_list: list[Project]) -> list[Project]:
  """ Download the summary result file from GCP bucket and add to dataclass """
  for project in project_list:
    os.makedirs(project.results_dir, exist_ok = True)
    !gsutil -q cp {project.sequencing_results_gcp} {project.results_dir}
    if not os.path.isfile(project.sequencing_results_local):
      raise Exception(f'Missing {project.name} sequencing_results csv.')
    else:
      project.results_df = pd.read_csv(project.sequencing_results_local,
                                      dtype=sequencing_results_dtype)
  return project_list

def make_qc_csv(project_list: list[Project]) -> pd.DataFrame:
  """ Make CSV file (qc_{date}.csv) to paste into the 'covid_seq_qc' Google sheet
  for Tableau visualization """
  print('making qc csv to add to gsheet')
  results = [p.results_df for p in project_list]
  concat_results = pd.concat(results)
  concat_results = concat_results[concat_results['sample_type'] != 'verification']

  qc_cols = ['hsn', 'sample_name', 'project_name', 'plate_name',
              'analysis_date', 'run_date', 'percent_coverage',
              'pangolin_lineage', 'expanded_lineage', 'aggregated_lineage',
              'mean_depth', 'mapped_reads', 'pangolin_version', 'platform',
              'sample_well', 'index_well', 'index_set', 'index_kit',
              'sample_type', 'hsn']
  #### NOTE ######
  # moving forward not all these columns will be available in the summary results
  # csv file. Will need to modify this list and gsheet to come
  qc_df = concat_results[qc_cols]
  qc_fn = os.path.join(qc_dir, f'qc_{date.today()}.csv')
  qc_df.to_csv(qc_fn, index=False, header = False)

  return qc_df

def append_new_results_to_gsheet(project_list: list[Project],
                                 qc_df: pd.DataFrame) -> None:
  """ Automatically append qc_df results to google sheet, but only if results
  have not already been appended """
  # get list of projects already in gsheet
  gs = gc.open('covid_seq_qc')
  worksheet = gs.get_worksheet(0)
  worksheet_df = pd.DataFrame.from_records(worksheet.get_all_values())
  worksheet_df.columns = worksheet_df.iloc[0]
  worksheet_df = worksheet_df.iloc[1:]
  projects_in_gsheet = worksheet_df.project_name.unique().tolist()

  # Append if needed
  for project in project_list:
    if project.name not in projects_in_gsheet:
        temp = qc_df[qc_df.project_name == project.name]
        values = temp.astype(str).replace('nan', 'null').replace('<NA>', 'null').values.tolist()
        gs.values_append('covid_seq_qc', {'valueInputOption': 'USER_ENTERED'}, {'values': values})
    else:
      print(f'{project.name} results have already been appended to the covid_seq_qc gsheet')
  return

def refresh_tableau_extact(datasource_id, tableau_auth):
  """ Refresh Tableau extract """
  print('Refreshing Tableau extract...')
  with server.auth.sign_in(tableau_auth):
    job = server.datasources.refresh(datasource_id)

    print(f'Update job posted. ID: {job.id}')
    print('Waiting for job...')
    job = server.jobs.wait_for_job(job)
    print('Job finished successfully')

    return

def pull_boxplot_qc_view(tableau_auth, server, view_id, projects):
  """ Download Tableau boxplot images """
  print('Getting Tableau boxplot...')
  image_path = os.path.join(tableau_dir, 'boxplot_view_image.png')
  project_str = ",".join(projects)
  with server.auth.sign_in(tableau_auth):
    i_req_option = TSC.ImageRequestOptions(imageresolution=TSC.ImageRequestOptions.Resolution.High)
    i_req_option.vf('Project Name', project_str)

    view = server.views.get_by_id(view_id)
    server.views.populate_image(view, i_req_option)

    # server.views.populate_image(view)
    with open(image_path, 'wb') as f:
      f.write(view.image)

  image = mpimg.imread(image_path)
  plt.figure(figsize=(10,10), frameon = False)
  plt.imshow(image)
  plt.show()

  return


# Download sequencing results from bucket, format for covid_seq_qc gsheet and append
project_list = download_summary_results(project_list)
qc_df = make_qc_csv(project_list)
results = [p.results_df for p in project_list]

if notebook_run_type == 'Production' and role == 'Analyst':
  # Append results to covid_seq_qc gsheet
  append_new_results_to_gsheet(project_list, qc_df)
  # Refresh Tableau datasource (covid_seq_qc (covid_seq_qc))
  refresh_tableau_extact(datasource_id = datasource_id, tableau_auth=tableau_auth)

# Print boxplots to screen
pull_boxplot_qc_view(tableau_auth, server, boxplot_view_id, projects)

# 👾✋✅ Compare S gene PCR against sequencing results

🧬 **S gene 69/70 deletion coordinates : 21,765 - 21,771** 🧬

After running this cell review the printed tables. **Based on the tables some samples will require manual review.** The tables summarize the count of samples that fall into `pass`, `fail` and `warn` categories as defined by the [s_gene_qc_truth_table](https://docs.google.com/spreadsheets/d/10mlS5me_lUbvmAFEVKcgwRmUQptsX9mHrP1LXSXYwms/edit?gid=874459759#gid=874459759). `pass` indicates no PCR/sequencing conflicts. `fail` indicates a PCR/sequencing conflict. `warn` indicates that there may be PCR/sequencing conflict (occurs when the N gene and S gene qPCR ct values are >3 ct apart indicating a partial S gene drop). See truth table for more details.

For fail and warn samples we check **1)** that the sample has >=90% coverage and **2)** that the genome region covering the 69/70 del in the s gene has greater than 20x depth. If a sample does not have >20x depth in the 69/70 del genome region then nextclade can not tell if the 69/70 del is present or absent; therefore samples with <20x depth in the 69/70 del can not be checked using the s gene truth table and should be ignored during this stage of QC.
   

How to interpret the tables:
1. Manual QC check table- this is the table that contains information on how many samples passed this notebook's QC checks, failed assembly,
or require manual review after automated S gene checks.
  - `qc_pass`: samples with no s gene PCR/sequencing conflicts, regardless of percent coverage. No manual review required.
  - `assembly_fail`: during assembly no consensus sequence was generated for these samples usually because of low or poor reads. No manual review required.
  - `fail_over_90_pc_over_20x_del`: samples with >=90% coverage and >=20x depth in the 69/70 del genome region that show a s gene PCR/sequening mismatch. **These samples require manual review. Samples will be printed below the tables and bam files will be located in the files to the left.**
  - `warn_over_90_pc_over_20x_del`: samples with >=90% coverage and >=20x depth in the 69/70 del genome region that show a s gene PCR/sequening mismatch. **These samples require manual review. Samples will be printed below the tables and bam files will be located in the files to the left.**
2. Other information that does not affect QC- this table contains information about the remaining S gene fails and warn samples. None of these meet the minimum requirements to check manually.
  - `fail_over_90_pc_under_20x_del`: samples with >=90% coverage and >=20x depth in the 69/70 del genome region that show a s gene PCR/sequening mismatch.
  - `warn_over_90_pc_under_20x_del`: samples with >=90% coverage and >=20x depth in the 69/70 del genome region that show a s gene PCR/sequening mismatch.
  - `fail_under_90_pc`: samples with < 90% coverage that show a s gene PCR/sequening mismatch.
  - `warn_under_90_pc`: samples with < 90% coverage that show a s gene PCR/sequening mismatch.

**After review, report any unresolved conflicts which may indicate a plate orientation issue or contamination issue to the qc checker.**


Resources:
- `s_gene_qc_truth_table` [google sheet](https://docs.google.com/spreadsheets/d/10mlS5me_lUbvmAFEVKcgwRmUQptsX9mHrP1LXSXYwms/edit?gid=874459759#gid=874459759)



In [ ]:
# @title 👾✋✅ Compare S gene PCR against sequencing results
## @markdown Run this cell and investigate all flagged samples suggesting an
## @markdown S gene PCR/sequencing inconsistencies.
## @markdown
## @markdown **Report any unresolved inconsistencies, suspected plate orientation
## @markdown issues, or PCR contamination issues to the QC checker.**
## @markdown
## @markdown Resources:
## @markdown - `s_gene_qc_truth_table` [google sheet](https://docs.google.com/spreadsheets/d/10mlS5me_lUbvmAFEVKcgwRmUQptsX9mHrP1LXSXYwms/edit?gid=874459759#gid=874459759)


def download_tableau_file(tableau_auth, server, view_id: str,
                          projects: list[str]) -> pd.DataFrame:
  """ Download tableau file from filtered view. """
  tableau_file = './tableau/tableau_covid_seq_qc_fview.csv'
  if not os.path.isfile(tableau_file):
    print('Downloading Tableau file...\n')
    project_str = ",".join(projects)

    with server.auth.sign_in(tableau_auth):
        csv_req_option = TSC.CSVRequestOptions()
        csv_req_option.vf('Project Name', project_str)

        view = server.views.get_by_id(view_id)
        server.views.populate_csv(view , csv_req_option)
        with open(tableau_file, 'wb') as f:
            f.write(b''.join(view.csv))
    print('Downloaded!')

  tableau_cols = ['HSN', 'Pangolin Lineage', 's_gene_ct', 'n_gene_ct',
                  'Day of Collect Date']
  tableau_df = (pd.read_csv(tableau_file,
                            dtype = {'HSN' : object},
                            usecols = tableau_cols)
                    .rename(columns={'HSN': 'hsn',
                                    'Pangolin Lineage': 'lineage',
                                    'Day of Collect Date': 'collection_date'}))

  # check to see if there are * in the df, this could indicate an error in tableau/dataset
  if tableau_df[tableau_df.hsn == '*'].shape[0] > 0:
    raise Exception('There are * in the tableau '
                    'rt-pcr_results_merged_with_seq_qc_table- something is '
                    'wrong. Check that hsn_paste column in covid_seq_qc gsheet '
                    'is formatted as plain text instead of as a number.')

  return tableau_df


def merge_pcr_seq_data(results: list[pd.DataFrame],
                       tableau_df: pd.DataFrame) -> pd.DataFrame:
  """Merge sequencing data (spike mutations, percent coverage) from
  `results` list of DataFrames with PCR data (Ct values) from `tableau_df`.
  """
  keep_cols = ['hsn', 'sample_name', 'project_name', 'spike_mutations',
              'percent_coverage', 'index_well', 'sample_type', 'assembly_pass']

  seq_results_df = (pd.concat(results)[keep_cols]
                    .query('sample_type == "sample"')
                    .astype({'hsn': str, 'spike_mutations': str})
                    .replace({'hsn': r'\.0$'}, {'hsn': ''}, regex=True)
                    .drop(columns = ['sample_type']))

  merged_pcr_seq_df = (seq_results_df
                       .merge(tableau_df, on='hsn', how = 'left')
                       .astype({'hsn': str, 'lineage': str}))

  return merged_pcr_seq_df


def run_qc(merged_pcr_seq_df: pd.DataFrame) -> pd.DataFrame:
  """Perform S gene QC on the `merged_pcr_seq_df` DataFrame and return
  the QC results in the `qc_output_df` DataFrame."""

  print('\nComparing s drop info against qc truth table...')
  qc_lookup_df = pd.DataFrame(gc
                    .open('s_gene_qc_truth_table')
                    .worksheet('lineage_free_s_gene_check')
                    .get_all_records()
                    )
  bool_cols = ['s_drop', 'partial_s_drop', 'has_69_70_deletion']
  qc_lookup_df[bool_cols] = (qc_lookup_df[bool_cols]
                             .replace({'TRUE': True, 'FALSE': False})
  )

  # Performing QC
  # Only on samples that pass assembly
  df = merged_pcr_seq_df.loc[merged_pcr_seq_df.assembly_pass]

  # Calculate qc comparison columns
  df['s_drop'] = (df.s_gene_ct == 0)
  df['partial_s_drop'] = (df.s_gene_ct >= df.n_gene_ct + 3)
  df['has_69_70_deletion'] = df.spike_mutations.str.contains(r'69del|70del')

  # Compare with qc table
  df = df.merge(qc_lookup_df,
                how = 'left',
                on = ['s_drop', 'partial_s_drop', 'has_69_70_deletion'])

  # Build assembly fail df to concat with assembly pass dfs
  assembly_fail = (
      merged_pcr_seq_df
      .loc[~merged_pcr_seq_df.assembly_pass]
      .assign(qc = 'assembly_fail')
  )
  assembly_fail[['s_drop','partial_s_drop',
                 'has_69_70_deletion', 'reason', 'action']] = None
  results_cols = ['hsn', 'sample_name', 'project_name', 'spike_mutations',
                  'percent_coverage', 'index_well', 'assembly_pass',
                  'collection_date', 'n_gene_ct', 'lineage', 's_gene_ct',
                  's_drop', 'partial_s_drop',
                  'has_69_70_deletion', 'qc', 'reason', 'action']
  assembly_fail = assembly_fail[results_cols]

  df_results = pd.concat([df, assembly_fail], ignore_index = True)

  df_depth = download_depth_bam_bai_files(df_results)
  qc_output_df = df_results.merge(df_depth, how = 'left')

  cols = ['hsn', 'sample_name', 'project_name', 'qc', 'reason', 'action',
          'lineage', 's_gene_ct', 'n_gene_ct', 'has_69_70_deletion',
          'percent_coverage', 'index_well', 'depth_69_70_deletion']
  qc_output_df = qc_output_df[cols].sort_values('project_name')

  return qc_output_df


def download_depth_bam_bai_files(df_results: pd.DataFrame) -> pd.DataFrame:
  """ Download depth.txt, bam, and bai files for qc fail/warn samples above 90%
  coverage and extract the 69/70 deletion depth """
  # Calculate paths for specific sample depth files to download all at once
  print('Downloading bam files...')
  df = df_results[(df_results.qc.isin(['fail', 'warn']) &
        (df_results.percent_coverage >= 90))][['sample_name', 'project_name']]
  df['bam_stats'] = df['bam_bai'] = ''

  return_cols = ['sample_name', 'project_name', 'depth_69_70_deletion']
  if df.shape[0] > 0:
    for proj in df.project_name.unique().tolist():
      project = list(filter(lambda p: p.name == proj, project_list))[0]
      df['bam_stats'] = np.where(df.project_name == proj,
                                project.terra_outputs + '/bam_stats/' +
                                df.sample_name + '_*_depth.txt',
                                df.bam_stats)
      df['bam_bai'] = np.where(df.project_name == proj,
                              project.terra_outputs + '/alignments/' +
                              df.sample_name + '*primertrimmed.rg.sorted.ba*',
                              df.bam_bai)

    # Write paths to txt file to pipe into gsutil cp
    bam_stats_paths_file = f'{samtools_depth_dir}/bam_stats_paths.txt'
    bam_bai_paths_file = f'{bam_dir}/bam_bai_paths.txt'
    df.bam_stats.to_csv(bam_stats_paths_file, index = False, header = False)
    df.bam_bai.to_csv(bam_bai_paths_file, index = False, header = False)

    # Download depth, bam, and bai files
    !cat {bam_stats_paths_file} | gcloud storage cp -I {samtools_depth_dir} --no-user-output-enabled
    !cat {bam_bai_paths_file} | gcloud storage cp -I {bam_dir} --no-user-output-enabled

    # Extract 69/70 del depth
    downloaded_depths = glob.glob(os.path.join(samtools_depth_dir,'*depth.txt'))
    df_depth_fns = pd.DataFrame(downloaded_depths, columns = ['depth_fn'])
    df_depth_fns['depth_fn_reg'] = (df_depth_fns
                                    .depth_fn
                                    .str.split('_')
                                    .str[:2]
                                    .str.join('_')
    )
    df['depth_fn_reg'] = samtools_depth_dir + df.sample_name
    df = df.merge(df_depth_fns, on = 'depth_fn_reg', how = 'left')
    if df.depth_fn.isna().any():
        print('Missing samtools depth files for the following samples:\n',
              df[df.depth_fn.isna()])

    df['depth_69_70_deletion'] = df.apply(lambda x: s_gene_deletion_depth(x.depth_fn),
                                          axis = 1)
  else:
    df = pd.DataFrame(columns = return_cols)
  return df[return_cols]


def s_gene_deletion_depth(depth_fn):
  """
  check the depth of the 69/70 del region (21,765)
  """

  try:
    depths = []
    for i in range(21765, 21771):
      depth = !grep {i} {depth_fn} | cut -f 3
      depths.append(int(depth[0]))
    return depths
  except IndexError:
    raise Exception('Depth not calculated correctly.')


def write_qc_results(projects: list[str], qc_output_df: pd.DataFrame) -> None:
  """Upload S gene QC results to GCP. Separate verification samples into their
  own df and output file. """
  print('Transferring results...')
  for project in projects:
    df = qc_output_df.query('project_name == @project')

    # reorder columns
    col_order = ['hsn', 'project_name', 'qc', 'reason', 'action', 'lineage',
                 's_gene_ct', 'n_gene_ct', 'has_69_70_deletion',
                 'percent_coverage', 'index_well', 'depth_69_70_deletion']
    df = df[col_order]

    out_fn = f'{project}_s_gene_qc_results.csv'
    local_out_path = os.path.join(qc_dir, out_fn)
    df.to_csv(local_out_path, index=False)
    gcp_destination = os.path.join(base_gcp_path_push, project, 'qc', out_fn)
    !gsutil -q cp {local_out_path} {gcp_destination}
  return


def over_90_qc_results(qc_output_df: pd.DataFrame) -> None:
  """Display QC failures and warnings for samples >= 90% coverage."""
  print('Generating summary QC results...')
  total_count = qc_output_df.shape[0]
  fail_df = (qc_output_df
             .query("qc != 'pass' & qc != 'assembly_fail'")
             .sort_values(by = ['project_name', 'hsn'])
  )

  under20 = 'max_depth < 20'
  over20 = 'max_depth >= 20'
  under90 = 'percent_coverage < 90'
  over90 = 'percent_coverage >= 90'
  fail = 'qc == "fail"'
  warn = 'qc == "warn"'

  fail_df['max_depth'] = fail_df.query(over90).depth_69_70_deletion.apply(max)

  # return fail/warn samples over 90% coverage and 20x coverage on 69/70 deletion
  qc_output_failures_df = (fail_df
                           .query(f'{over90} & {over20}')
                           .drop(columns = ['max_depth'])
  )

  # Get number of samples per project in qc status, 90 percent coverage, and 20x
  # depth categories
  dfs_info_inspect, dfs_info_other = [], []

  # Check over 90% coverage samples
  for dc, dc_str in {over20: 'over_20x_del', under20: 'under_20x_del'}.items():
    for qc, qc_str in {fail: 'fail', warn: 'warn'}.items():
      col =  '_'.join([qc_str, 'over_90_pc', dc_str])
      query_df = (fail_df
                  .query(f'{over90} & {qc} & {dc}')
                  [['hsn', 'project_name']]
                  .groupby(['project_name'])
                  .count()
                  .rename(columns = {'hsn' : col})
      )
      # Fill out info for any projects that didn't have samples in this category
      if query_df.shape[0] < len(projects):
        empty_df = pd.DataFrame(index = projects, columns = [col]).fillna(0)
        query_df = (pd
                    .concat([query_df.copy(), empty_df])
                    .reset_index()
                    .drop_duplicates(subset = 'index', keep = 'first')
                    .set_index('index')
                    .rename_axis('project_name')
                    )
      if dc_str == 'over_20x_del':
        dfs_info_inspect.append(query_df)
      else:
        dfs_info_other.append(query_df)

  # Check under 90% coverage samples
  for qc, qc_str in {fail: 'fail', warn: 'warn'}.items():
    col =  '_'.join([qc_str, 'under_90_pc'])
    query_df = (fail_df
                .query(f'{under90} & {qc}')
                [['hsn', 'project_name']]
                .groupby(['project_name'])
                .count()
                .rename(columns = {'hsn' : col})
    )
    if query_df.shape[0] < len(projects):
      empty_df = pd.DataFrame(index = projects, columns = [col]).fillna(0)
      query_df = (pd
                  .concat([query_df.copy(), empty_df])
                  .reset_index()
                  .drop_duplicates(subset = 'index', keep = 'first')
                  .set_index('index')
                  .rename_axis('project_name')
                  )
    dfs_info_other.append(query_df)

  # Get qc pass and assembly fail info
  passes = (qc_output_df
            .query("qc == 'pass'")[['hsn', 'project_name']]
            .groupby(['project_name'])
            .count()
            .rename(columns = {'hsn' : 'qc_pass'})
            )
  assembly_fails = (qc_output_df
                    .query("qc == 'assembly_fail'")[['hsn', 'project_name']]
                    .groupby(['project_name'])
                    .count()
                    .rename(columns = {'hsn' : 'assembly_fail'})
                    )
  dfs_info_inspect.insert(0, assembly_fails)

  df_info_inspect = passes.join(dfs_info_inspect).fillna(0).astype('Int64')
  df_info_other = dfs_info_other[0].join(dfs_info_other[1:]).fillna(0).astype('Int64')
  df_info_inspect.loc['TOTAL'] = df_info_inspect.sum()
  df_info_other.loc['TOTAL'] = df_info_other.sum()

  print(f'\nChecked {total_count} samples.')
  print('\nManual QC check table:')
  display(df_info_inspect)

  print('\n\nOther information that does not affect QC:')
  display(df_info_other)

  return qc_output_failures_df


# Download tableau data and merge with results
tableau_df = download_tableau_file(tableau_auth,
                                   server,
                                   view_id = rt_pcr_view_id,
                                   projects = projects)

merged_pcr_seq_df = merge_pcr_seq_data(results, tableau_df)

# Assign S gene QC results to each sample
qc_output_df = run_qc(merged_pcr_seq_df)

# push results to bucket and display on screen
write_qc_results(projects, qc_output_df)
qc_output_failures_df = over_90_qc_results(qc_output_df)

if not qc_output_failures_df.empty:
  # Add info for proportion of lineages with S drop in case it's not close to 0 or 1
  qc_output_failures_df = qc_output_failures_df.sort_values(by = ['qc', 'project_name'])
  print(f'{Colors.RED}\nQC FAILURES / WARNINGS:{Colors.ENDC}')
  pd.set_option('display.max_colwidth', None)
  display(qc_output_failures_df)

# 👾✋✅ Check results summary before posting to GSP chat

Run this cell and check over the output before posting it into the GSP chat.

Check the following and report any critera not met to the QC checker.
- Positive control is > 80% coverage*
- Positive control lineage matches the expected lineage (B.1.617.1 as of 7/23/25)*
- Negative control is < 10%
- At least 40 samples have > 90% coverage
- No suspcious lineages

*If the positive control is < 80% coverage, the run may still pass. To determine if the run passes the number of samples with > 90% coverage and the number of samples passing the s gene qc truth table check (cell above) are taken into account. When this occurs a warning statement in red will be printed below the positive control that lists the number of samples categorized as `pass`, `warn` and `fail` based on the s gene qc truth table (See cell above). Be sure to update the sample counts in each category based on any resolved `warn` and `fail` samples from the cell above.

*Positive controls with too low a perecent coverage may have an unassigned lineage.



<!-- # @title 👾✋✅ Check results summary before posting to GSP chat
# @markdown Run this cell and check over the output before posting to the GSP chat.
# @markdown
# @markdown **Check the following and report any failures to the QC checker.**
# @markdown
# @markdown *   ✔ Positive control is > 80% coverage and matches the lineage of
# @markdown the control used during sequencing.
# @markdown *   ✔ Negative control is < 10%.
# @markdown *   ✔ At least 40 samples have > 90% coverage.
# @markdown *   ✔ If any lineages look suspicious
# @markdown
# @markdown
# @markdown Note on 7/23/25- The lineage of the positive control is currently
# @markdown B.1.617.1. The code will tell you if the positive control lineage
# @markdown does not match this. If it doesn't match, ask sequencing if they have
# @markdown updated the control. If they have, please update the variable
# @markdown `pos_current_lineage` in the function `check_pos_control`. -->

In [ ]:
# @title 👾✋✅ Check results summary before posting to GSP chat
## @markdown Run this cell and check over the output before posting to the GSP chat.
## @markdown
## @markdown **Check the following and report any failures to the QC checker.**
## @markdown
## @markdown *   ✔ Positive control is > 80% coverage and matches the lineage of
## @markdown the control used during sequencing.
## @markdown *   ✔ Negative control is < 10%.
## @markdown *   ✔ At least 40 samples have > 90% coverage.
## @markdown *   ✔ If any lineages look suspicious
## @markdown
## @markdown
## @markdown Note on 7/23/25- The lineage of the positive control is currently
## @markdown B.1.617.1. The code will tell you if the positive control lineage
## @markdown does not match this. If it doesn't match, ask sequencing if they have
## @markdown updated the control. If they have, please update the variable
## @markdown `pos_current_lineage` in the function `check_pos_control`.

def get_all_previous_lineages() -> set:
  """ Get set of all lineages already sequenced in earlier runs """
  # Get list of lineages from covid_seq_qc sheet
  sheet = gc.open('covid_seq_qc')
  all_df = pd.DataFrame(sheet.get_worksheet(0).get_all_records())
  all_df = all_df[~all_df.project_name.isin(projects)]
  all_lineages = set(all_df.pangolin_lineage.unique())

  # Also get set of lineages from the old 'QC Runs' sheet (stored in a pickle)
  legacy_lineages_pickle = os.path.join(f'{shared_drive_path}/COVID/Bioinformatics/legacy_all_lineages.pickle')
  with open(legacy_lineages_pickle, 'rb') as handle:
    legacy_lineages = pickle.load(handle)

  all_lineages = all_lineages.union(legacy_lineages)
  return all_lineages

def get_collection_date_ranges(tableau_df: pd.DataFrame, df: pd.DataFrame) -> str:
  """ Get dictionary with range of collection dates for each project """
  collection_dates_df = tableau_df[['hsn', 'collection_date']]
  collection_dates_df['collection_date'] = pd.to_datetime(collection_dates_df['collection_date'])

  merged_df = collection_dates_df.merge(df, on = 'hsn')
  merged_df['collection_date'] = pd.to_datetime(merged_df['collection_date'])

  earliest_date = merged_df['collection_date'].min().strftime('%Y-%m-%d')
  latest_date = merged_df['collection_date'].max().strftime('%Y-%m-%d')
  collection_ranges = f'Collected {earliest_date} to {latest_date}'

  return collection_ranges

def check_pos_control(row: pd.Series, qc_df: pd.DataFrame) -> str:
  """ QC check the positive control """
  pos_current_lineage = 'B.1.617.1'
  pos_name = row['sample_name']
  pos_cov = row['percent_coverage']
  pos_lineage = row['pangolin_lineage']
  pos_line = f'{pos_name} = {pos_cov}% cov ({pos_lineage})'
  if pos_cov < 80:
    txt = f'{Colors.RED}WARNING: **** {pos_line} ****{Colors.ENDC}'
  else:
    txt = pos_line

  if (pos_lineage != pos_current_lineage) and (pos_cov > 0):
    if pos_lineage == 'Unassigned':
        pos_warn = (f'\n{Colors.RED}WARNING: **** {pos_name} lineage '
                    f'was unassigned.****{Colors.ENDC}')
    else:
        pos_warn = (f'\n{Colors.RED}WARNING: **** {pos_name} lineage '
                    f'({pos_lineage}) doesn\'t match expected lineage of '
                    f'{pos_current_lineage}. ****{Colors.ENDC}')

    txt += pos_warn
  return txt

def get_controls_coverage(df: pd.DataFrame, qc_df: pd.DataFrame) -> str:
  """ Get NC and POS coverage """
  NC = df[df['sample_name'].str.contains('NTC*|NC*')]
  if NC.shape[0] > 0:
    row = NC.iloc[0]
    nc_name = row['sample_name']
    nc_cov = row['percent_coverage']
    if nc_cov >= 10:
      nc_line = f'{Colors.RED}WARNING: **** {nc_name} = {nc_cov}% cov ****{Colors.ENDC}'
    else:
      nc_line = f'{nc_name} = {nc_cov}% cov'
  else:
    nc_line =  f'{Colors.RED}WARNING: **** No negative control found ****{Colors.ENDC}'

  POS = df[df['sample_name'].str.contains('POS*')].reset_index(drop = True)
  if POS.shape[0] > 0:
    row = POS.iloc[0]
    pos_line = check_pos_control(row, qc_df)
  else:
    pos_line = f'{Colors.RED}WARNING: **** No positive control found ****{Colors.ENDC}'

  return nc_line, pos_line

def get_sample_coverage(df: pd.DataFrame) -> str:
  """ Get sample coverage"""
  df = df.query('sample_type == "sample" & percent_coverage >= 90')
  if df.shape[0] > 40:
    line = f'\n{df.shape[0]} seqs have >= 90% cov'
  else:
    line = f'\n{Colors.RED}WARNING: *** {df.shape[0]} seqs have >= 90% cov ***{Colors.ENDC}'
  return line

def get_samples_with_unassigned_lineages(df: pd.DataFrame) -> str:
  """ Get unassigned lineage count"""
  df = df.query('sample_type == "sample" & percent_coverage >= 90 & pangolin_lineage == "Unassigned"')
  cov_line = 'Unassigned lineages with > 90% cov:'
  if df.shape[0] > 0:
    df['line_list'] = df.sample_name + ': ' + df.percent_coverage.astype(str) + '% cov'
    str_of_samples = '\n'.join(df.line_list)
    line = f'{Colors.RED}WARNING *** {cov_line} ***\n{str_of_samples}{Colors.ENDC}'
  else:
    line = f'{cov_line}\nNONE'

  return line

def get_new_lineages(df: pd.DataFrame) -> str:
  """ Get new lineages that have not been seen before"""
  new_lineages_df = (df[~df['pangolin_lineage'].isin(all_prev_lineages) &
                        ~df['pangolin_lineage'].isnull() &
                        (df['pangolin_lineage'] != 'Unassigned')]
                    .sort_values(by = 'pangolin_lineage'))

  keep_cols = ['hsn', 'pangolin_lineage', 'percent_coverage']
  new_lineages_table = new_lineages_df[keep_cols].to_string(index=False, header=False)

  if new_lineages_df.shape[0] > 0:
    line = f'New lineages:\n{new_lineages_table}'
  else:
    line = f'New lineages:\nNONE'

  return line

def get_aggregrated_lineage_counts(df: pd.DataFrame) -> str:
  """ Count number of samples belonging to each aggregated_lineage for samples
  with >90% coverage """
  df = df.query('sample_type == "sample" & percent_coverage >= 90')
  counts = df.groupby(['aggregated_lineage'])['aggregated_lineage'] \
              .count().sort_values(ascending=False)

  lineages_str = '\n'.join("{}: {}".format(k, v) for k, v in counts.items())

  line = f'Aggregated lineage counts:\n{lineages_str}'
  return line

def print_results_summary(project: Project) -> None:
  """print results summary to screen
  example of print out:

  cov_2169_grid
  Collected 2024-06-27 to 2024-07-31
  NC-cov_2169_grid = 0.0% cov
  POS-cov_2169_grid = 89.68% cov (B.1.1.369)
  76 seqs have >= 90% cov
  unassigned lineages with > 90% cov = NONE

  New lineages:
  None

  Aggregated lineages:
  KP.3: 25
  KP.3.1.1: 13
  KP.2.3: 11
  LB.1: 6
  KP.1.1: 5
  KP.1.1.3: 4
  KS.1: 4
  JN.1: 3
  KP.2: 3
  JN.1.18: 1
  LP.1: 1


  *** Molly to QC ***
  ----------------------------------
  """
  # project line
  df = project.results_df
  qc_df = qc_output_df.query('project_name == @project.name')
  print('\033[1m' + project.name + '\033[0m')
  print(get_collection_date_ranges(tableau_df, df))
  print(*get_controls_coverage(df, qc_df), sep = '\n')
  print(get_sample_coverage(df), '\n')
  print(get_samples_with_unassigned_lineages(df), '\n')
  print(get_new_lineages(df), '\n')
  print(get_aggregrated_lineage_counts(df), '\n')
  print(f'*** {qc_checker} to QC ***\n\n----------------------------------\n')

  return


all_prev_lineages = get_all_previous_lineages()
print('\033[1m' + today + '\033[0m' + '\n')
for project in project_list:
  print_results_summary(project)
# print WGS Horizon Reports bucket
print(f'https://drive.google.com/drive/folders/1w7vWrND_opr9rvi0d9LxfAbv_FWcBNsg')

---
# 🕒 Section 3 : QC Check by QC checker
**This section should be completed by the QC checker, who will need to rerun the notebook begining from the top.**

In this section, you will confirm sequencing results and QC performed by the analyst, push a QC report to the GCP project bucket, generate the parsing files for Horizon, add failed samples to the rerun samples sheet, and add projects to submissions ready list.

<br>

### ✋✅ CHECKPOINT: Confirm Sequencing Results and QC performed by Analyst
The analyst should communicate any QC fails or other concerns to you. Perform further investigation when necessary.

In [ ]:
# @title 👾 Select failed projects/samples
# @markdown To select multiple projects/samples from the drop down menus hold down the ctrl key.

possible_project_failures = widgets.SelectMultiple(
  options=projects,
  description='Choose any failed projects:',
  disabled=False,
  style = {'description_width' : 'initial'}
)

possible_sample_failures_dict = (qc_output_failures_df.groupby('project_name')
                                  .hsn.apply(list).to_dict())
possible_sample_failures = widgets.SelectMultiple(
  options=qc_output_failures_df.hsn.tolist(),
  description='Choose any failed samples:',
  disabled=False,
  style = {'description_width' : 'initial'},
  layout=widgets.Layout(width='400px', height='300px'))


remaining_samples = sorted(list(set(qc_output_df.sample_name.unique().tolist()) -
                    set(qc_output_failures_df.hsn.tolist())))

pcr_mixup_samples = widgets.SelectMultiple(
  options=remaining_samples,
  description='Choose any possible PCR mixup samples:',
  disabled=False,
  style = {'description_width' : 'initial'},
  layout=widgets.Layout(width='400px', height='300px'))

items = [possible_project_failures, possible_sample_failures, pcr_mixup_samples]
box = widgets.Box(children=items)
display(box)


In [ ]:
# @title 👾 Finalize failed projects/samples
box.close()
failed_projects = list(possible_project_failures.value)
passed_projects = list(set(projects) - set(failed_projects))
failed_samples = list(possible_sample_failures.value) + list(pcr_mixup_samples.value)

increase_font(17)
if projects_failed:= (len(failed_projects) > 0):
  print(f'\n{Colors.RED}Failed projects:{Colors.ENDC}',
        *(p for p in failed_projects),
        sep = '\n')
  print(f'{Colors.OKGREEN}Passed projects:{Colors.ENDC}',
        *(p for p in passed_projects),
        sep = '\n')
else: print(f'{Colors.OKGREEN}All projects marked as passed{Colors.ENDC}')
print()


if len(failed_samples) > 0:
  print(f'{Colors.RED}Failed samples:{Colors.ENDC}',
        *(s for s in failed_samples),
        sep = '\n')
  print(f'{Colors.OKGREEN}\nAll other samples marked as passed{Colors.ENDC}')
else: print(f'{Colors.OKGREEN}All samples in passed projects marked as passed{Colors.ENDC}')

sample_failures_dict = (qc_output_df
                       [qc_output_df.sample_name.isin(failed_samples)]
                       .groupby('project_name')
                       .hsn.apply(list).to_dict()
                       )

In [ ]:
# @title 👾 Update run_tracker, push output files, and delete fastqs for failed projects

def update_run_tracker() -> None:
  """ Update qc_status and qc_note column in run_tracker. Asks for user input on
  the reason for any project failures. """

  for p in projects:
    cell = run_tracker.find(p)
    if p in failed_projects:
      qc_note = input(f'Marking {p} as qc fail on the run_tracker. Please type the reason why it failed: ')
      run_tracker.update_cell(cell.row, cell.col + 7, 'fail')
      run_tracker.update_cell(cell.row, cell.col + 8, qc_note)
    else:
      run_tracker.update_cell(cell.row, cell.col + 7, 'pass')


def write_qc_results(failed_projects: list[str],
                     sample_failures_dict: dict,
                     project_list: list[Project]) -> None:
  """ Write and push final qc results files """
  print(f'\nTransferring QC report files to {base_gcp_path_push} for projects:')
  for project in project_list:
    qc_results_df = project.results_df[project.results_df['sample_type'] !=
                                    'verification'][['hsn', 'sample_name']].copy()

    # Fill QC column based on failed projects and failed samples
    qc_results_df['qc'] = 'pass'  # initialize all to QC pass

    if project.name in failed_projects:
      qc_results_df['qc'] = 'fail'
      print(f'{project.name} - Marked all samples in {project.name} as "fail"')
    elif project.name in sample_failures_dict.keys():
      samples = sample_failures_dict[project.name]
      qc_results_df.loc[qc_results_df['hsn'].isin(samples), 'qc'] = 'fail'
      print(f'{project.name} - Marked sample(s) {", ".join(samples)} as "fail" '
            'and the remaining as "pass"')
    else:
      print(f'{project.name} - Marked all samples in {project.name} as "pass"')

    # Add qc column to result CSV and write to qc directory
    qc_results_csv = os.path.join(qc_dir, f'{project.name}_final_qc_results.csv')
    qc_results_df['project_name'] = project.name
    qc_results_df.to_csv(qc_results_csv, index=False)

    destination = f'{base_gcp_path_push}{project.name}/qc/'
    !gsutil -q cp {qc_results_csv} {destination}

def copy_result_files_to_shared_drive(project_list: list[Project]):
  print(f'\nTransferring results files to {results_drive_path} for projects:')
  for project in project_list:
    print(project.name)
    results_project_path = os.path.join(results_drive_path, project.name)
    !rm -rf {project.results_dir}/.ipynb_checkpoints  # Don't copy over checkpoint folder
    !cp -R {project.results_dir}/* "{results_project_path}"
  return

def make_batch_number_dict():
  """ Get dict of {project_name : batch_numer} from run_tracker """
  have_batch_num = [i for i in run_tracker.get('C:D') if len(i) == 2]
  batch_dict = dict(have_batch_num)
  return batch_dict

def make_horizon_csvs(project_list: list[Project]) -> None:
  """ Make and push CSV file for QC pass projects to parse sequencing results
  into Horizon """
  cols = ['hsn', 'percent_coverage', 'mean_depth', 'pangolin_lineage',
          'aggregated_lineage', 'expanded_lineage', 'pangolin_version',
          'project_name', 'platform', 'analysis_date', 'run_date']
  print(f'\nTransferring Horizon parsing files to {horizon_drive_path} for projects:')
  for project in project_list:
    if project.name in passed_projects:
      try:
        batch = batch_dict[project.name]
        filename = os.path.join(horizon_report_dir, f'MOL_{batch}_dmax.csv')
        print(project.name, f'(MOL_{batch})')
      except KeyError:
        print(project.name)
        print(f'{project.name} - note- The batch number for {project.name} '
              'hasn\'t been filled out in the run tracker, so sequencing will '
              'have to rename the file manually.')
        filename = os.path.join(horizon_report_dir, f'{project.name}_wgs_horizon_report_v2.csv')

      df = project.results_df.copy()
      df = df[df['sample_type'] != 'verification'][cols].sort_values('hsn')
      df.to_csv(filename, index = False)
      !cp {filename} "{horizon_drive_path}/"
    else:
      print(f'{project.name} - marked as failed, so no Horizon parsing file created')
  return

def remove_failed_project_fastqs(project_list: list[Project]):
  for project in project_list:
    if project.name in failed_projects:
      print(f'Removing raw and scrubbed fastqs for failed project {project.name}...')
      !gsutil -q -m rm -r {project.raw_fastqs_gcp}
      !gsutil -q -m rm -r {project.scrubbed_fastqs_gcp}

if role == 'QC checker':
  if notebook_run_type == 'Production':
    update_run_tracker()

  # push qc results to bucket
  write_qc_results(failed_projects, sample_failures_dict, project_list)

  # copy result files to shared drive
  copy_result_files_to_shared_drive(project_list)

  # make horizon parsing file and copy to shared drive
  batch_dict = make_batch_number_dict()
  make_horizon_csvs(project_list)

  if notebook_run_type == 'Production':
    remove_failed_project_fastqs(project_list)

else:
  print(f'{Colors.RED}Are you the QC checker? If so, make sure to update your '
        'role at the top of the notebook to update the run_tracker. If not, '
        f'you shouldn\'t be running this section of the notebook.{Colors.ENDC}')

In [ ]:
# @title 👾✋✅ Print final QC for GSP chat
for project in projects:
  if project in failed_projects:
    print(f'{Colors.RED}{Colors.BOLD}{project} = QC FAIL{Colors.ENDC}')
  elif project in sample_failures_dict.keys():
    print(f'{Colors.OKGREEN}{project} = QC PASS,{Colors.RED} except for the '
          f'following sample failure(s):{Colors.BOLD}')
    samples = sample_failures_dict[project]
    print(*('\t- ' + s + '\n' for s in samples), f'{Colors.ENDC}', sep = '')
  else:
    print(f'{Colors.OKGREEN}{project} = QC PASS{Colors.ENDC} 🦛🦛🦛🦛🦛')

In [ ]:
# @title 👾 Final steps: Add analyst name to ready submissions and add failed samples to `manual_sample_exclusions` sheet
def update_submission_records(passed_projects: list[str]) -> None:
  # add analyst name to submissions ready list for QC pass projects
    submission_sheet = gc.open('SUBMISSION_records').worksheet('clinical_not_submitted')
    values_list = submission_sheet.col_values(1)
    for i, v in enumerate(values_list):
      if v in passed_projects:
        submission_sheet.update_cell(i + 1, 2, analyst)

    print(f'Added {analyst} to clinical_not_submitted tab for projects {", ".join(passed_projects)}')
    return

def add_sample_reruns(sample_failures_dict: dict) -> None:
  # Add individual failed samples to sample_reruns sheet with QC reason
  if len(sample_failures_dict) > 0:
    # Switch the keys/values to make it easier to add to rerun samples sheet
    sample_failures_dict2 = {i: k for (k, v) in sample_failures_dict.items()
                            for i in v}
    rerun_sheet = gc.open(sample_exclusions).worksheet('cov')
    for k, v in sample_failures_dict2.items():
      try:
        exclusion_note = str(qc_output_failures_df.query('sample_name == @k')['reason'].iloc[0])
      except IndexError:
        exclusion_note = 'Potential PCR plate mixup'
      hsn = str(qc_output_df.query('sample_name == @k')['hsn'].iloc[0])
      new_row = [hsn, k, v, 'rerun - qc failure', exclusion_note]
      rerun_sheet.append_row(new_row, table_range = 'A1')
    print(f'Added {len(failed_samples)} sample(s) to rerun_samples sheet')

  else: print(f'No failed samples to add to rerun_samples sheet')
  return

if notebook_run_type == 'Production':
    update_submission_records(passed_projects)

add_sample_reruns(sample_failures_dict)
